# r/soccer topic modelling on Google Colab (GPU)

**FIFA Communities chapter — BERTopic pipeline (`code/05-topic-model.py`) run on a Colab GPU.**

This notebook runs *exactly the same code* as on the local Mac; only the device changes (`--device cuda`).
Use it for the full corpus (17 year models, 5 tournament windows, 5 era alignments) and for LLM labelling
with the larger Qwen3-4B model.

**Before running:**
1. In Colab: *Runtime → Change runtime type → GPU* (T4 is fine; L4/A100 with Colab Pro is faster).
2. Upload to your Google Drive, in a folder `fifa-reddit-communities/`:
   * `code/` (the whole folder: `topicmodel/`, `05-topic-model.py`, `requirements-topicmodel.txt`),
   * `data/yearly datasets/` (the 17 yearly CSVs you need, ~2.4 GB in total; one file per year is enough for a single year run).
3. Results are written to `fifa-reddit-communities/results/topic-models/<run-name>/` on Drive; copy them back
   into the repository's `results/` folder afterwards.

Colab sessions can disconnect (≈12 h limit on the free tier). Embeddings and models are cached per year on Drive,
and LLM labelling is resumable (`labels.jsonl`), so simply re-run the cell that stopped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/fifa-reddit-communities'   # adjust if you used another folder name
import os; assert os.path.exists(f'{ROOT}/code/05-topic-model.py'), 'upload the code/ folder first'
os.chdir(ROOT); print('working in', os.getcwd())

In [ ]:
# Install the pinned environment (2–4 minutes). hdbscan compiles from source on Colab.
!pip install -q -r code/requirements-topicmodel.txt
import torch; print('CUDA available:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 1. One year model (example: 2014)
Each run writes `results/topic-models/year-<YYYY>/` with `report.html`, `topics.csv`, `documents.csv`, topics over time,
sentiment tables, the merge table and the saved model. Add `--label` to label topics in the same run (slower), or label
afterwards with the `--label-only` cell below. Use `--embedding-model sentence-transformers/all-mpnet-base-v2` for the
higher-quality embeddings now that a GPU is available (the cache is keyed by model name).

In [ ]:
EMB = 'sentence-transformers/all-mpnet-base-v2'   # or 'sentence-transformers/all-MiniLM-L6-v2' to match the local test
!HF_HOME=$ROOT/.hf-cache python code/05-topic-model.py --run year --year 2014 --device cuda --embedding-model $EMB

## 2. All years, tournament windows and eras
Run the loop below (it skips runs that already have a `report.html`). Expect roughly 5–15 minutes per year on a T4
with mpnet; windows are faster; era alignment needs the four year models of that era first.

In [ ]:
import os, subprocess
EMB = 'sentence-transformers/all-mpnet-base-v2'
def run(args):
    print('>>>', args, flush=True)
    subprocess.run(f'HF_HOME={ROOT}/.hf-cache python code/05-topic-model.py {args} --device cuda --embedding-model {EMB}', shell=True, check=True)
for y in range(2010, 2027):
    if not os.path.exists(f'results/topic-models/year-{y}/report.html'): run(f'--run year --year {y}')
for wc in (2010, 2014, 2018, 2022, 2026):
    if not os.path.exists(f'results/topic-models/window-wc{wc}/report.html'): run(f'--run window --worldcup {wc}')
for era in (2010, 2014, 2018, 2022):          # 2026 has a single year; no alignment needed
    if not os.path.exists(f'results/topic-models/era-{era}/config.json'): run(f'--run era --era {era}')

## 3. LLM labelling (two-analyst debate, Qwen3-4B)
Resumable: re-run the same cell after a disconnect and it continues from `labels.jsonl`. About 10–20 s per topic on a T4.

In [ ]:
RUN = 'results/topic-models/year-2014'
!HF_HOME=$ROOT/.hf-cache python code/05-topic-model.py --label-only $RUN --llm Qwen/Qwen3-4B --device cuda

## 4. After co-author review: apply the merge table
Edit `topic_merge_table.csv` of a run (column `merge_into`, `final_name`), then:

In [ ]:
!python code/05-topic-model.py --run year --year 2014 --apply-merges